# Bilancia and co

In [ ]:
import pyvisa
import numpy as np
import time
import serial
from serial.tools import list_ports
import matplotlib.pyplot as plt
import cv2

In [ ]:
%matplotlib widget
plt.close('all')

In [ ]:
class Bilancia:

    comandi = {
        "identificazione" : (chr(255) + chr(254) + chr(1) + chr(1) + chr(0) + chr(254)).encode('latin-1'),
        "iniziazione misure" : (chr(255) + chr(254) + chr(1) + chr(5) + chr(2) + chr(248) + chr(1) + chr(255)).encode('latin-1'),
        "stop" : (chr(255)+chr(254)+chr(1)+chr(5)+chr(2)+chr(0)+chr(0)+chr(248)).encode('latin-1')
    }

    def __init__(self, porta, baudrate=9600, timeout=2):
        self.porta = porta
        self.baudrate = baudrate
        self.timeout = timeout
        self.libra = serial.Serial(self.porta, self.baudrate, timeout=self.timeout)
    
    def reset_buffers(self):
        self.libra.reset_input_buffer()
        self.libra.reset_output_buffer()
    
    def send_command(self, comando):
        if comando in self.comandi:
            self.libra.write(self.comandi[comando])
        else:
            raise ValueError(f"Comando '{comando}' non riconosciuto.")

    def identify(self):
        self.send_command("identificazione")
        risposta = self.libra.read(60)
        return risposta.decode('latin-1')
    
    def start_measurements(self):
        self.send_command("iniziazione misure")
        self.libra.read(8) #leggo correta ricezione del messaggio di prendere le misure
    
    def read_measurement(self):
        self.libra.read(5) #leggo la testa della singola misura
        rate_string = self.libra.read(5) #something like b'...'
        spessore_string = self.libra.read(6)    
        self.libra.read(32)# leggo la coda della singola misura

        rate = rate_string.decode('latin-1')
        spessore=spessore_string.decode('latin-1')
        
        return float(rate), float(spessore)
    
    def stop_measurements(self):
        self.reset_buffers()
        self.send_command("stop")
        self.reset_buffers()

        self.libra.close()
    

class Multimetro:

    comandi = {
        "identificazione": '*IDN?',
        "reset": '*RST',
        "imposta in ohm": 'OHMS',
        "autoscale": "AUTO",
        "leggi valore1": 'VAL1?',

    }   

    def __init__(self, resource_manager, resource_name):
        self.resource_manager = resource_manager
        self.multimeter = self.resource_manager.open_resource(resource_name)
    
    def identify(self):
        self.multimeter.write(self.comandi["identificazione"])
        response = self.multimeter.read()
        self.multimeter.read() # pulisce il buffer
        return response

    def initialize(self):
        self.multimeter.write(self.comandi["reset"])
        time.sleep(2)
        self.multimeter.read() # pulisce il buffer
        self.multimeter.write(self.comandi["autoscale"])
        time.sleep(2)
        self.multimeter.read() # pulisce il buffer
        self.multimeter.write(self.comandi["imposta in ohm"])
        self.multimeter.read() # pulisce il buffer
    
    def read_value(self):
        self.multimeter.write(self.comandi["leggi valore1"])
        value = self.multimeter.read()
        self.multimeter.read() # pulisce il buffer
        return float(value)

    def close(self):
        self.multimeter.close()


class Camera:
    def __init__(self, camera_index=0):
        self.cap = cv2.VideoCapture(camera_index)

        self.im0 = None  # Immagine di riferimento

        if not self.cap.isOpened():
            raise RuntimeError("Impossibile aprire la webcam.")
    
    def set_camera_params(self, exposure=-5, wb_temp=3900):
        self.cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
        self.cap.set(cv2.CAP_PROP_EXPOSURE, exposure)
        self.cap.set(cv2.CAP_PROP_AUTO_WB, 0)
        self.cap.set(cv2.CAP_PROP_WB_TEMPERATURE, wb_temp)
    
    def acquire_image(self):
        ret, frame = self.cap.read()
        if not ret:
            raise RuntimeError("Impossibile acquisire un frame dalla webcam.")
        
        return frame
    
    def build_roi_masks(self, im0, center_x, center_y, radius):
        h, w = im0.shape[:2]
        Y, X = np.ogrid[:h, :w]
        dist_from_center = np.sqrt((X - center_x)**2 + (Y - center_y)**2)
        
        masks = {
            'total': dist_from_center <= radius,
            'mid': dist_from_center <= (radius / 2),
            'in': dist_from_center <= (radius / 4),
            'q1': (dist_from_center <= radius) & (Y < center_y) & (X > center_x),
            'q2': (dist_from_center <= radius) & (Y < center_y) & (X < center_x),
            'q3': (dist_from_center <= radius) & (Y > center_y) & (X < center_x),
            'q4': (dist_from_center <= radius) & (Y > center_y) & (X > center_x)
        }
        
        return masks
    
    def acquire_reference_image(self, avgs=16):
        frames = []
        for _ in range(avgs):
            frame = self.acquire_image()
            frames.append(frame.astype(np.float32))
            time.sleep(0.1) # Breve pausa per dare tempo al sensore
            
        if not frames:
            raise RuntimeError("Acquisizione immagine di riferimento fallita.")
            
        self.im0 = np.mean(frames, axis=0).astype(np.uint8)
        self.im0 = cv2.cvtColor(self.im0, cv2.COLOR_BGR2GRAY)
        
        return self.im0.astype(np.uint8)

    def process_frame(self, im0, masks):

        if im0 is None or not masks:
            raise ValueError("Immagine di riferimento e maschere ROI devono essere inizializzate prima di processare.")

        frame = self.acquire_image()
        im1_float = frame.astype(np.float32)
        diff = im1_float - im0
        heatmap = np.linalg.norm(diff, axis=-1)

        ii = np.mean(255 - diff[masks['total']])
        ii_in = np.mean(255 - diff[masks['in']])
        ii_mid = np.mean(255 - diff[masks['mid']])
        
        ii1 = np.mean(255 - diff[masks['q1']])
        ii2 = np.mean(255 - diff[masks['q2']])
        ii3 = np.mean(255 - diff[masks['q3']])
        ii4 = np.mean(255 - diff[masks['q4']])
        
        return frame, heatmap, ii, ii_in, ii_mid, ii1, ii2, ii3, ii4

    def release(self):
        self.cap.release()
    
    def __delete__(self, instance):
        self.release()



In [ ]:
if False:
    rm = pyvisa.ResourceManager()
    inst = rm.list_resources()  #chiediamo quali porte sono aperte

    coms = list_ports.comports()
    print("Porte VISA disponibili:")
    print(inst)
    print("\nPorte seriali disponibili:")
    [print(i, end='; ') for i in coms]

""

In [ ]:
camera = Camera(camera_index=0)
camera.set_camera_params()
im0 = camera.acquire_reference_image(avgs=16)

plt.figure()
plt.imshow(im0, cmap='gray')
plt.colorbar()
plt.show()

camera.release()